In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.stem import PorterStemmer
import pickle

In [2]:
movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')

In [3]:
credits.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [4]:
credits['crew'][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

## Dataset Columns

### movies.csv — Columns We Use
- `genres` → Action, Drama, Sci-Fi etc
- `keywords` → story-related words like dream, space, heist
- `overview` → short plot description
- `vote_average` → movie rating out of 10

### credits.csv — Columns We Use
- `cast` → list of actors
- `crew` → we only extract the **director** name

### Columns We Ignore
- `budget`, `revenue`, `homepage`, `runtime` → not useful for recommendations

In [5]:
# merging both datasets on title
movies = movies.merge(credits, on='title')
movies.shape

(4809, 23)

In [6]:
# keeping only useful columns
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'release_date']]

movies.head()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,2007-05-19
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,2015-10-26
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,2012-07-16
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,2012-03-07


In [7]:
# checking for missing values
movies.isnull().sum()

movie_id        0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
release_date    1
dtype: int64

In [8]:
movies.dropna()

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,2007-05-19
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,2015-10-26
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,2012-07-16
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,2012-03-07
...,...,...,...,...,...,...,...,...,...
4804,9367,El Mariachi,El Mariachi just wants to play his guitar and ...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 5616, ""name"": ""united states\u2013mexi...","[{""cast_id"": 1, ""character"": ""El Mariachi"", ""c...","[{""credit_id"": ""52fe44eec3a36847f80b280b"", ""de...",6.6,1992-09-04
4805,72766,Newlyweds,A newlywed couple's honeymoon is upended by th...,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",[],"[{""cast_id"": 1, ""character"": ""Buzzy"", ""credit_...","[{""credit_id"": ""52fe487dc3a368484e0fb013"", ""de...",5.9,2011-12-26
4806,231617,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam...","[{""cast_id"": 8, ""character"": ""Oliver O\u2019To...","[{""credit_id"": ""52fe4df3c3a36847f8275ecf"", ""de...",7.0,2013-10-13
4807,126186,Shanghai Calling,When ambitious New York attorney Sam is sent t...,[],[],"[{""cast_id"": 3, ""character"": ""Sam"", ""credit_id...","[{""credit_id"": ""52fe4ad9c3a368484e16a36b"", ""de...",5.7,2012-05-03


In [9]:
movies.dtypes

movie_id          int64
title            object
overview         object
genres           object
keywords         object
cast             object
crew             object
vote_average    float64
release_date     object
dtype: object

In [10]:
movies['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [11]:
# converting string to list and extracting only names
def convert(text):
    l = []
    for i in ast.literal_eval(text):
        l.append(i['name'])
    return l

In [12]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)

## Converting Genres and Keywords

when we printed genres[0] it was a string that looked like a list
it had id and name for each genre but we only need the name

so wrote a function called convert() that does 2 things
1. converts the string into a real python list using ast.literal_eval()
2. loops through each item and extracts only the 'name' value

applied this function to both genres and keywords columns
now instead of a messy string we get a clean list like ['Action', 'Adventure']

In [13]:
movies['cast'][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [14]:
# extracting only top 3 cast members
def convert3(text):
    l = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3:
            l.append(i['name'])
            counter += 1
    return l

movies['cast'] = movies['cast'].apply(convert3)

## Extracting Top 3 Cast Members

the cast column had 80+ actors for each movie which is too much
we only need the main actors (top 3) because they define the movie's identity
for example knowing its Sam Worthington and Zoe Saldana tells more than 80 random names

so wrote a function convert3() that
1. converts the string into a real python list using ast.literal_eval()
2. loops through the list but stops after 3 names using a counter

applied this to the cast column
now each movie has only the top 3 actor names as a clean list

In [15]:
# extracting only the director from crew
def fetch_director(text):
    l = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            l.append(i['name'])
    return l

movies['crew'] = movies['crew'].apply(fetch_director)

## Extracting Director from Crew

the crew column had hundreds of people like producers, editors, sound designers etc
we only need the director because the director defines the style and feel of the movie

so wrote a function fetch_director() that
1. converts the string into a real python list using ast.literal_eval()
2. loops through each crew member and checks if their job == 'Director'
3. if yes, extracts only their name

applied this to the crew column
now each movie has only the director name as a list

In [16]:
# converting overview from string to list
movies['overview'] = movies['overview'].fillna('').apply(lambda x: x.split())

In [17]:
movies['overview'][0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.']

In [18]:
# combining all columns into one tags column
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [19]:
movies['tags'][0]

['In',
 'the',
 '22nd',
 'century,',
 'a',
 'paraplegic',
 'Marine',
 'is',
 'dispatched',
 'to',
 'the',
 'moon',
 'Pandora',
 'on',
 'a',
 'unique',
 'mission,',
 'but',
 'becomes',
 'torn',
 'between',
 'following',
 'orders',
 'and',
 'protecting',
 'an',
 'alien',
 'civilization.',
 'Action',
 'Adventure',
 'Fantasy',
 'Science Fiction',
 'culture clash',
 'future',
 'space war',
 'space colony',
 'society',
 'space travel',
 'futuristic',
 'romance',
 'space',
 'alien',
 'tribe',
 'alien planet',
 'cgi',
 'marine',
 'soldier',
 'battle',
 'love affair',
 'anti war',
 'power relations',
 'mind and soul',
 '3d',
 'Sam Worthington',
 'Zoe Saldana',
 'Sigourney Weaver',
 'James Cameron']

In [20]:
movies.columns

Index(['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew',
       'vote_average', 'release_date', 'tags'],
      dtype='object')

In [21]:
new_df = movies[['movie_id', 'title', 'tags', 'vote_average', 'release_date']]
new_df.head()

,movie_id,title,tags,vote_average,release_date
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...",7.2,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, d...",6.9,2007-05-19
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, send...",6.3,2015-10-26
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney...",7.6,2012-07-16
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, mili...",6.1,2012-03-07


In [22]:
new_df = movies[['movie_id', 'title', 'tags', 'vote_average', 'release_date']].copy()
new_df['tags'] = new_df['tags'].apply(lambda x: ' '.join(x))

In [23]:
new_df.head()

,movie_id,title,tags,vote_average,release_date
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",7.2,2009-12-10
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",6.9,2007-05-19
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,6.3,2015-10-26
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,7.6,2012-07-16
4,49529,John Carter,"John Carter is a war-weary, former military ca...",6.1,2012-03-07


In [24]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [25]:
ps = PorterStemmer()

def stem(text):
    l = []
    for i in text.split():
        l.append(ps.stem(i))
    return ' '.join(l)

new_df['tags'] = new_df['tags'].apply(stem)

In [26]:
new_df['tags'][0]

'in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between follow order and protect an alien civilization. action adventur fantasi scienc fiction cultur clash futur space war space coloni societi space travel futurist romanc space alien tribe alien planet cgi marin soldier battl love affair anti war power relat mind and soul 3d sam worthington zoe saldana sigourney weaver jame cameron'

## Lowercasing and Stemming the Tags

after merging all columns into tags, two more cleaning steps were done

first converted everything to lowercase so that 'Action' and 'action' 
are not treated as two different words by the vectorizer

then applied stemming using PorterStemmer from nltk
stemming reduces words to their root form
for example 'running', 'runs', 'ran' all become 'run'
'loving' becomes 'love', 'fighting' becomes 'fight'

this makes sure similar words are counted as one word
which improves the accuracy of our recommendations

In [27]:
# converting tags into vectors
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags']).toarray()
vectors.shape

(4809, 5000)

## Vectorization using CountVectorizer

before this step all the movie info was in a text format (tags column)
but machine learning can only work with numbers, not text

so used CountVectorizer to convert the tags into numbers
it works in 2 steps:
1. builds a vocabulary of top 5000 most common words across all movies
2. for each movie, counts how many times each word appears

the result is a matrix of shape (4809, 5000)
meaning 4809 movies, each represented as a row of 5000 numbers
now every movie is a vector that the ML can understand

used 2 extra settings:
- max_features=5000 → only keep top 5000 words to keep it efficient
- stop_words='english' → automatically removes useless words like 'the', 'a', 'is'

## CountVectorizer Output (Vectors)

CountVectorizer counts how many times each word appears in the tags of each movie
the result is a table where rows are movies and columns are words from the vocabulary

| Movie | action | drama | fight | hero | love | romance | thriller |
|-------|--------|-------|-------|------|------|---------|----------|
| Avatar | 2 | 0 | 1 | 1 | 0 | 0 | 0 |
| Titanic | 0 | 2 | 0 | 0 | 3 | 2 | 0 |
| Inception | 1 | 1 | 0 | 0 | 0 | 0 | 2 |

values are whole numbers showing how many times each word appears
0 means that word is not present in that movie's tags

In [28]:
# calculating cosine similarity between all movies
similarity = cosine_similarity(vectors)
similarity.shape

(4809, 4809)

In [29]:
similarity

array([[1.        , 0.06622662, 0.06554735, ..., 0.04442617, 0.03244428,
        0.        ],
       [0.06622662, 1.        , 0.0412393 , ..., 0.0372678 , 0.        ,
        0.01712972],
       [0.06554735, 0.0412393 , 1.        , ..., 0.01844278, 0.0808122 ,
        0.        ],
       ...,
       [0.04442617, 0.0372678 , 0.01844278, ..., 1.        , 0.03651484,
        0.03064257],
       [0.03244428, 0.        , 0.0808122 , ..., 0.03651484, 1.        ,
        0.06713451],
       [0.        , 0.01712972, 0.        , ..., 0.03064257, 0.06713451,
        1.        ]])

## Cosine Similarity Matrix

after vectorization every movie is a row of 5000 numbers
now we need to find which movies are similar to each other

used cosine_similarity() to calculate similarity between all movie vectors
it compares every movie with every other movie and gives a score between 0 and 1

- score = 1.0 → movies are identical (same movie compared to itself)
- score close to 0 → movies are very different
- higher the score → more similar the movies

the result is a matrix of shape (4809, 4809)
meaning every movie is compared with all 4809 movies

this matrix is the core of our recommendation system
when a user selects a mood → we find similarity scores against that mood
sort from highest to lowest → top 10 are the recommendations

## Cosine Similarity Output (Similarity Matrix)

after vectorization, cosine similarity calculates the angle between every two movie vectors
and converts it into a score between 0 and 1

| Movie | Avatar | Titanic | Inception |
|-------|--------|---------|-----------|
| Avatar | 1.0 | 0.02 | 0.31 |
| Titanic | 0.02 | 1.0 | 0.08 |
| Inception | 0.31 | 0.08 | 1.0 |

diagonal is always 1.0 because every movie is identical to itself
higher the score → more similar the two movies are
this matrix is used to find the best matching movies for each mood

In [30]:
# mood mapping - our unique feature
mood_tags = {
    'Mind Bending': "scifi thriller mystery dream twist mind psychology illusion reality",
    'Adventure': "action adventure fantasy epic hero journey quest warrior",
    'Romance': "love romance drama relationship emotion heart passion wedding",
    'Comedy': "comedy funny humor laugh lighthearted fun entertaining silly",
    'Edge of Seat': "horror suspense dark crime thriller violence scary tension",
    'Emotional': "sad drama grief loss family emotional deep touching heart"
}

# function to recommend movies based on mood
def recommend_by_mood(mood):
    # convert mood tags to vector
    mood_vector = cv.transform([mood_tags[mood]]).toarray()
    
    # calculate similarity between mood vector and all movies
    mood_similarity = cosine_similarity(mood_vector, vectors)[0]
    
    # sort movies by similarity score
    movies_list = sorted(list(enumerate(mood_similarity)), reverse=True, key=lambda x: x[1])
    
    # return top 10
    print(f"Top 10 movies for mood: {mood}")
    for i in movies_list[1:11]:
        print(new_df.iloc[i[0]].title, "→ score:", round(i[1], 3))

### The code is just doing:

Create fake movie string for each mood ✅
Convert it to vector using same CountVectorizer ✅
Find most similar real movies using cosine similarity ✅
Return top 10 ✅



In [31]:
recommend_by_mood('Adventure')

Top 10 movies for mood: Adventure
The Hobbit: An Unexpected Journey → score: 0.218
The Forbidden Kingdom → score: 0.217
Last Action Hero → score: 0.199
The Good Dinosaur → score: 0.192
Dragonball Evolution → score: 0.192
Batman v Superman: Dawn of Justice → score: 0.189
The Helix... Loaded → score: 0.189
Amidst the Devil's Wings → score: 0.189
The Hebrew Hammer → score: 0.183
Superhero Movie → score: 0.182


In [32]:
recommend_by_mood('Mind Bending')

Top 10 movies for mood: Mind Bending
Mulholland Drive → score: 0.226
Yesterday Was a Lie → score: 0.211
A Nightmare on Elm Street 5: The Dream Child → score: 0.195
Duplex → score: 0.187
Banshee Chapter → score: 0.181
The Lawnmower Man → score: 0.178
When the Lights Went Out → score: 0.17
Transcendence → score: 0.164
Insidious: Chapter 3 → score: 0.163
Listening → score: 0.162


In [33]:
recommend_by_mood('Emotional')

Top 10 movies for mood: Emotional
Four Single Fathers → score: 0.227
A Mighty Heart → score: 0.223
Live-In Maid → score: 0.218
Hesher → score: 0.216
Death Sentence → score: 0.214
Blood Work → score: 0.208
As It Is in Heaven → score: 0.204
Boynton Beach Club → score: 0.204
Rabbit Hole → score: 0.2
Poetic Justice → score: 0.197


In [34]:
recommend_by_mood('Romance')

Top 10 movies for mood: Romance
Closer → score: 0.314
August Rush → score: 0.312
Eternal Sunshine of the Spotless Mind → score: 0.312
Crazy/Beautiful → score: 0.307
Heavenly Creatures → score: 0.29
Veer-Zaara → score: 0.286
Aimee & Jaguar → score: 0.283
The Bridges of Madison County → score: 0.281
The Last Song → score: 0.279
The Incredibly True Adventure of Two Girls In Love → score: 0.272


## Mood Mapping — Our Unique Feature

this is the part that makes our project different from the original

created a dictionary called mood_tags where each mood has a set of keywords
that describe the feeling and genre of that mood
for example Adventure has keywords like action, hero, journey, quest etc

when a user selects a mood, the recommend_by_mood() function does 4 things:

1. gets the keyword string for that mood from the dictionary
2. converts that string into a vector using the same CountVectorizer we built earlier
   (we use transform not fit_transform because vocabulary is already built)
3. calculates cosine similarity between the mood vector and all 4809 movie vectors
   this gives us a score for every movie showing how close it is to the selected mood
4. sorts all movies by score from highest to lowest and returns top 10

this way instead of searching by movie name
the user just selects a mood and gets the most relevant movies for that feeling

In [36]:
# mood vector table
mood_vector_df = pd.DataFrame()

for mood, tags in mood_tags.items():
    vector = cv.transform([tags]).toarray()[0]
    mood_vector_df[mood] = vector

mood_vector_df.index = cv.get_feature_names_out()

# show only words that have non-zero values
mood_vector_df = mood_vector_df[mood_vector_df.sum(axis=1) > 0]
print(mood_vector_df)

              Mind Bending  Adventure  Romance  Comedy  Edge of Seat  \
action                   0          1        0       0             0   
adventure                0          1        0       0             0   
comedy                   0          0        0       1             0   
crime                    0          0        0       0             1   
dark                     0          0        0       0             1   
deep                     0          0        0       0             0   
drama                    0          0        1       0             0   
dream                    1          0        0       0             0   
epic                     0          1        0       0             0   
family                   0          0        0       0             0   
fun                      0          0        0       1             0   
grief                    0          0        0       0             0   
heart                    0          0        1       0          

## How Cosine Similarity Finds Movies for a Mood

### Step 1 — Mood Vector (our fake movie)

| | action | adventure | hero | love | romance | horror |
|--|--------|-----------|------|------|---------|--------|
| Adventure Mood | 1 | 1 | 1 | 0 | 0 | 0 |

### Step 2 — Real Movie Vectors

| | action | adventure | hero | love | romance | horror |
|--|--------|-----------|------|------|---------|--------|
| The Hobbit | 1 | 1 | 1 | 0 | 0 | 0 |
| Titanic | 0 | 0 | 0 | 1 | 1 | 0 |
| Batman | 1 | 1 | 0 | 0 | 0 | 0 |
| The Conjuring | 0 | 0 | 0 | 0 | 0 | 1 |

### Step 3 — Cosine Similarity Score (angle between mood and each movie)

| Movie | Shared words with Adventure Mood | Score |
|-------|----------------------------------|-------|
| The Hobbit | action ✅ adventure ✅ hero ✅ | 0.31 |
| Batman | action ✅ adventure ✅ hero ❌ | 0.21 |
| Titanic | action ❌ adventure ❌ hero ❌ | 0.02 |
| The Conjuring | action ❌ adventure ❌ hero ❌ | 0.01 |

### Step 4 — Sort Highest to Lowest → Top 10 = Recommendations

| Rank | Movie | Score |
|------|-------|-------|
| 1 | The Hobbit | 0.31 ✅ |
| 2 | Batman | 0.21 ✅ |
| 3 | Titanic | 0.02 ❌ |
| 4 | The Conjuring | 0.01 ❌ |

more shared words = smaller angle = higher score = recommended

In [ ]:
# saving the model files
import pickle


pickle.dump(new_df, open('../model/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('../model/similarity.pkl', 'wb'))
pickle.dump(cv, open('../model/cv.pkl', 'wb'))

## Saving the Model with Pickle

saved 3 files to the model/ folder:
- movie_list.pkl → cleaned movie dataframe
- similarity.pkl → cosine similarity matrix
- cv.pkl → CountVectorizer

pickle saves python objects to disk so the streamlit app 
can directly load them without running the notebook again